In [0]:
# FIX: build_pipeline was missing entirely — this caused FIELD_NOT_FOUND: features
# because no stage was ever assembling raw columns into a 'features' vector.
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA


def build_pipeline(df, use_pca=False):
    """
    Builds preprocessing stages for a Spark ML Pipeline.
    Assembles raw columns -> scales -> optionally applies PCA.
    Always outputs a column named 'features' for the model stage.
    """
    feature_cols = [c for c in df.columns if c not in ["label", "weight"]]

    assembler = VectorAssembler(
        inputCols=feature_cols,
        outputCol="raw_features"
    )

    scaler = StandardScaler(
        inputCol="raw_features",
        outputCol="scaled_features",
        withMean=True,
        withStd=True
    )

    if use_pca:
        pca = PCA(
            k=10,
            inputCol="scaled_features",
            outputCol="features"
        )
        return [assembler, scaler, pca]
    else:
        # Rename scaled_features -> features so model always sees 'features'
        renamer = VectorAssembler(
            inputCols=["scaled_features"],
            outputCol="features"
        )
        return [assembler, scaler, renamer]

In [0]:
# from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier, LinearSVC


# def build_model(model_type, params, use_weight=False):

#     weight_col = "weight" if use_weight else None

#     if model_type == "lr":
#         return LogisticRegression(
#             labelCol="label",
#             featuresCol="features",
#             weightCol=weight_col,
#             **params
#         )

#     elif model_type == "rf":
#         return RandomForestClassifier(
#             labelCol="label",
#             featuresCol="features",
#             weightCol=weight_col,
#             **params
#         )

#     elif model_type == "gbt":
#         # GBTClassifier does not support weightCol in Spark ML — omitted intentionally
#         return GBTClassifier(
#             labelCol="label",
#             featuresCol="features",
#             **params
#         )

#     elif model_type == "svm":
#         # FIX: LinearSVC does not support weightCol — passing it caused a runtime crash.
#         # Removed weightCol from LinearSVC.
#         return LinearSVC(
#             labelCol="label",
#             featuresCol="features",
#             **params
#         )

from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier,
    LinearSVC
)

# from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier, LinearSVC


def build_model(model_type, params, use_weight=False):

    base = dict(labelCol="label", featuresCol="features")
    
    # Only add weightCol if use_weight is True
    if use_weight:
        base["weightCol"] = "weight"

    if model_type == "lr":
        return LogisticRegression(**base, **params)

    elif model_type == "rf":
        return RandomForestClassifier(**base, **params)

    elif model_type == "gbt":
        # GBTClassifier does not support weightCol — remove it even if use_weight=True
        base.pop("weightCol", None)
        return GBTClassifier(**base, **params)

    elif model_type == "svm":
        # LinearSVC does not support weightCol
        base.pop("weightCol", None)
        return LinearSVC(**base, **params)